# 2026-03-17 (화) - Tokens & Embeddings

**W2 Day 1: 토큰, 임베딩, 청킹 그리고 코사인 유사도**

오늘 배울 내용:
- **토큰(Token)**: LLM이 텍스트를 처리하는 최소 단위 (BPE 방식)
- **임베딩(Embedding)**: 고차원 텍스트를 저차원 벡터로 압축한 표현
- **청킹(Chunking)**: 긴 문서를 모델 컨텍스트 윈도우에 맞게 자르기
- **코사인 유사도(Cosine Similarity)**: 벡터 간 의미적 유사도 측정
- **CacheBackedEmbeddings**: 임베딩 API 비용 절감 캐시

핵심 비유:
- 임베딩 = 모델이 학습한 **"세계관"** 속 좌표
- 토큰 = 이 세계관의 **"글자 부품"**
- 코사인 유사도 = 두 좌표 사이의 **"방향 일치도"**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag/llm_260317_Tokens_Embeddings.ipynb)

## 0. Colab 환경 설정

Colab에서 실행 시 필요한 패키지 설치 및 API 키 로드.

In [ ]:
# Colab 환경에서 필요한 라이브러리 설치
# (로컬/이미 설치된 환경이면 건너뛰어도 됨)
!pip install -q langchain-openai langchain-text-splitters tiktoken scikit-learn langchain_classic

In [ ]:
# 기본 라이브러리 임포트
# os, json, time: 시스템 / 파일 / 시간 관련 유틸
# numpy, pandas: 행렬 & 테이블 데이터 처리
# matplotlib: 시각화
# Counter: 단순 카운팅 도구
# dotenv: .env 파일에서 환경변수 로드 (로컬용)
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv

# Colab에서는 userdata, 로컬에서는 .env 사용
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except ImportError:
    load_dotenv()
    api_key = os.getenv('OPENAI_API_KEY')

## 1. LLM & Embedding 모델 초기화

### 임베딩(Embedding)이란?

> **비유**: 사람의 머릿속 '개념 지도'를 생각해 보자. '사과'와 '바나나'는 가깝고, '사과'와 '자동차'는 멀다. 모델도 단어/문장을 **1536차원 공간의 한 점**으로 찍는다. 이 점의 좌표가 바로 임베딩 벡터.

- **고차원 데이터(텍스트/이미지/영상/음성)** → **저차원 벡터 스페이스로 압축된 표현**
- 이미지 예시: `(3, 1024, 1024)` RGB 이미지 → `(32, 32)` 저차원 매트릭스
- 텍스트 예시: `(100 시퀀스, 768 차원, 50000 voca)` → 1536차원 벡터
- 이 벡터 공간 = 모델의 **"세계관(representation space / manifold)"**

In [ ]:
# LangChain에서 OpenAI Chat 모델과 Embedding 모델 클래스 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
# LLM: 답변 생성용 (gpt-4o-mini - 저렴 & 빠름)
# Embeddings: 텍스트 -> 벡터 변환용 (text-embedding-3-small, 1536차원)
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)

In [ ]:
# 간단히 "Hello" 단어를 임베딩 해보기
# embed_query: 단일 텍스트 -> 벡터 하나
test_emb = embeddings_model.embed_query("Hello world")
len(test_emb)  # 1536 -> 이 모델은 1536차원짜리 벡터를 반환

In [ ]:
# 앞쪽 10개 값만 슬쩍 구경
# 소수점 형태의 알 수 없는 숫자들 - 이게 "Hello"의 좌표다
test_emb[:10]

## 2. Tokenizer - tiktoken 으로 토큰 들여다보기

### 토큰(Token)이란?

LLM이 텍스트를 처리하는 **최소 단위**. API 요금도 토큰 단위로 청구된다.
- gpt-4o-mini: 1M input = $0.05 / 1M output = $0.2

**자연어 → (토크나이저) → 숫자 ID 리스트 → (임베딩 모델) → 벡터**

- **encode**: 텍스트 → 토큰 ID 리스트
- **decode**: 토큰 ID 리스트 → 텍스트

In [ ]:
# tiktoken: OpenAI가 만든 BPE 토크나이저 라이브러리
# !pip install tiktoken  # Colab에서 미리 설치되어 있음
import tiktoken

In [ ]:
# gpt-4o-mini 모델의 인코더(토크나이저)를 가져옴
# 이 인코더가 바로 "gpt-4o-mini의 세계관"에서 사용하는 토큰 사전
enc = tiktoken.encoding_for_model('gpt-4o-mini')

In [ ]:
# 한글 문장을 토큰화 해보기
text = '안녕하세요. 오늘 LLM에 대해 배워볼게요!'
tokens = enc.encode(text)  # 텍스트 -> 토큰 ID 리스트
tokens

In [ ]:
# 디코드: 토큰 ID 리스트 -> 원래 텍스트
# 인코드-디코드가 완전히 일치(lossless)함을 확인
enc.decode(tokens)

In [ ]:
# 각 토큰이 실제로 어떤 글자 조각에 해당하는지 하나씩 확인
# -> 한글 "안녕하세요"는 '안' + '녕하세요' 로 두 토큰으로 쪼개짐!
# -> '배워볼게요' 같은 단어도 '배'/'워'/'볼'/'게'/'요' 식으로 잘게 나뉨
for i, token_id in enumerate(tokens):
    token_text = enc.decode([token_id])
    print(f' token {i+1} : ID : {token_id} -> {token_text}')

### 왜 한국어는 불규칙하게 쪼개질까? - BPE (Byte Pair Encoding)

> **비유**: 레고 블록 조립. 자주 같이 붙어 쓰이는 작은 블록들은 하나의 큰 블록으로 합쳐서 사전에 등록. 덜 쓰이는 조합은 낱개로 남김.

**과거 방식 (단어 단위)** - OOV(Out-Of-Vocabulary) 문제:
```
I like go to school -> I / like / go / to / school  (사전에 있음 → OK)
He likes go to school -> He / <unk> / go / to / school  (likes 사전에 없음!)
```

**현대 LLM 방식 (BPE, Byte Pair Encoding)**:
- 처음엔 모든 글자를 낱개로 쪼갬: `안 / 녕 / 하 / 세 / 요`
- 말뭉치를 돌면서 자주 붙어 나오는 쌍을 합쳐나감: `안녕 / 하세요`
- 계속 반복하면 빈도 높은 덩어리가 하나의 토큰이 됨
- 장점: `likes`가 사전에 없어도 `like` + `s`로 분해 가능 → OOV 해결

**결과**:
- 한글 1글자 = 약 2~3 토큰 (BPE가 영어 말뭉치 중심으로 학습됨)
- 영어 1단어 = 약 1~2 토큰
- → 한글은 같은 글자 수라도 API 비용이 더 비싼 경향

## 3. Text Splitter - 긴 문서 청킹

### 왜 문서를 자르나?

- LLM에는 **Context Window** 제한이 있음 (한 번에 받는 입력 크기 한계)
- 사내 매뉴얼 1권, 제품 사양서 PDF 100페이지 → 한 번에 못 넣음
- 그래서 **Chunk** 단위로 쪼개서 검색하여 필요한 부분만 LLM에 전달

### Splitter 종류

- **CharacterTextSplitter**: 문자(character) 개수 기준
- **RecursiveCharacterTextSplitter**: 문단/문장/단어 계층을 보면서 재귀적으로 쪼갬 (권장)
- **from_tiktoken_encoder**: 토큰 개수 기준 (모델 비용/윈도우와 직접 매칭)

### 핵심 파라미터
- `chunk_size`: 한 청크의 최대 크기 (문자 or 토큰)
- `chunk_overlap`: 앞뒤 청크가 겹치는 분량 (문맥 보존용) - chunk_size보다 작게!
- `separators`: 자를 우선순위 리스트 `["\n\n", "\n", ". ", " ", ""]`

> **비유**: 김밥을 자를 때 무작정 길이로 자르면 계란 한가운데가 잘린다. `RecursiveCharacterTextSplitter`는 "재료 경계(문단 → 문장 → 단어)"를 우선 살려가며 자른다.

In [ ]:
# 실습용 긴 텍스트 (4개 문단)
long_text = """인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다."""

In [ ]:
# Colab에서는 설치가 필요한 경우
!pip install -q langchain_text_splitters

In [ ]:
# RecursiveCharacterTextSplitter: 문단/문장 구조를 보며 재귀적으로 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# chunk_size=100(문자), chunk_overlap=20(앞뒤 겹침 20자)
# separators 우선순위: 문단(\n\n) > 줄바꿈(\n) > 문장(. ) > 단어(공백) > 글자 하나
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_text(long_text)
len(chunks)  # 총 청크 개수

In [ ]:
# chunk_size를 100에서 50으로 줄이면? 더 잘게 쪼개지고 문장 중간에서도 끊김
# 오버랩(20자)이 앞뒤 청크 문맥을 이어주는 걸 확인
splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=20,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_text(long_text)
for i, chunk in enumerate(chunks):
    print(f'[chunk {i+1}] : {len(chunk)} characters')
    print(chunk)

### 토큰 기반 청킹 - `from_tiktoken_encoder`

문자 수가 아니라 **실제 토큰 수**로 자르면 모델 비용/컨텍스트 예산과 정확히 맞출 수 있다.

In [ ]:
# 텍스트의 실제 토큰 개수를 세는 유틸 함수
def count_token(text, model='gpt-4o-mini'):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [ ]:
# from_tiktoken_encoder: 토큰 단위로 청킹
# chunk_size=50 = 50 토큰 (문자 수는 더 많을 수 있음!)
splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='gpt-4o-mini', chunk_size=50, chunk_overlap=10
)
chunks = splitter_tiktoken.split_text(long_text)
for i, chunk in enumerate(chunks):
    tokens = count_token(chunk)
    print(f'[chunk {i+1}] : {len(chunk)} characters, 실제 토큰: {tokens} tokens')
    print(chunk)

### 실습: 한국어 텍스트 청킹 리포트 함수 만들기

`chunk_size` / `chunk_overlap`을 입력받아 청크 개수, 평균/최대 토큰 등을 출력.

In [ ]:
# 한국어 test_text를 chunking 해보세요. 다양한 chunk size, overlap size를 이용
test_text = """서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다."""

In [ ]:
# [내 풀이] 캐릭터 단위 & 토큰 단위 둘 다 실험
# 캐릭터(자수) 단위로 나누기
splitter_char = RecursiveCharacterTextSplitter(
    chunk_size=50, chunk_overlap=10, separators=['\n\n', '\n', ', ', ' ', '']
)
chunks_char = splitter_char.split_text(test_text)
for i, chunk in enumerate(chunks_char):
    print(f'[chunk {i+1}] : {len(chunk)} characters')
    print(chunk)
print(' ========== ')

# 토큰 단위로 나누기
splitter_token = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='gpt-4o-mini', chunk_size=100, chunk_overlap=10
)
chunks_token = splitter_token.split_text(test_text)
for i, chunk in enumerate(chunks_token):
    tokens = count_token(chunk)
    print(f'[chunk {i+1}] : {len(chunk)} characters, 실제 토큰: {tokens} tokens')
    print(chunk)

In [ ]:
# [선생님 답안] 청킹 결과를 요약 리포트로 출력하는 함수
def split_and_report(text, chunk_size=50, chunk_overlap=10):
    # 토큰 기반 splitter 초기화
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        model_name='gpt-4o-mini', chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_text(text)
    # 각 청크의 실제 토큰 수 카운트
    token_counts = [count_token(c) for c in chunks]

    print(f'chunking report (chunk_size = {chunk_size}, chunk_overlap = {chunk_overlap})')
    # zip(chunks, token_counts): 청크와 토큰수를 한 쌍으로 묶어 순회
    for i, (chunk, tc) in enumerate(zip(chunks, token_counts)):
        first_line = chunk.split('\n')[0][:40]
        print(f' [{i+1}] {tc} tokens, {len(chunk)} characters | {first_line}')

    print(f'summary : {len(chunks)} chunks')
    print(f'average : {np.mean(token_counts):.2f}')
    print(f'max : {max(token_counts)}')

In [ ]:
# 함수 실행: test_text를 50 토큰 / 10 오버랩으로 청킹 리포트
split_and_report(test_text)

## 4. Embedding - 문장을 벡터로 변환

### 임베딩 모델 = 모델의 세계관

> **비유**: 1536차원의 공간에 "사과", "바나나", "자동차" 같은 모든 단어/문장을 점으로 찍어둔 지도. "사과↔바나나"는 가까이, "사과↔자동차"는 멀리 떨어져 있음.

- `embed_query(text)`: 단일 텍스트 → 1차원 벡터 (길이 1536)
- `embed_documents([t1, t2, ...])`: 여러 텍스트 → 2차원 벡터 리스트
- **벡터의 차원은 모델이 결정한다** (1536은 text-embedding-3-small의 고정값)
- 차원이 크면 표현력이 풍부, 작으면 빠르고 저렴

**프로세스**:
```
"인공지능..." -(tokenizer)-> [토큰 ID 리스트] -(embedding model)-> [1536차원 벡터]
```

In [ ]:
# 임베딩 모델 재임포트 (혹시 위에서 커널 다시 띄웠을 경우 대비)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
# text-embedding-3-small 모델로 초기화 (1536차원, 비용 저렴)
embedding_model = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)
# embedding_model('hello')  # 주의: 객체를 직접 호출하면 안됨. embed_query/embed_documents 써야 함

In [ ]:
# 쿼리 하나를 벡터로 변환
query = '인공지능이 세상을 바꾸고 있습니다.'
query_vector = embeddings_model.embed_query(query)
len(query_vector)  # 1536

### 차원이 크면 뭐가 좋을까?

- **15000차원**: 주어↔목적어, 주어↔다음 단어 예측, 단어↔동사 관계 등 **풍부한 관계** 표현 가능
- **1536차원**: 주어, 목적어, 동사 정도의 기본 관계만 표현
- **작은 차원**: 연산량 ↓, 속도 ↑, 비용 ↓ / 표현력 ↓
- **큰 차원**: 연산량 ↑, 속도 ↓, 비용 ↑ / 표현력 ↑

실무에선 트레이드오프를 보고 선택.

In [ ]:
# 여러 문서를 한 번에 임베딩 (embed_documents)
text = [
    "AI가 세상을 바꾸고 있습니다.",
    'AI는 인공지능입니다.',
    '머신러닝으로 질병을 예측할 수 있다.',
    '치킨 먹고 싶어요'
]

doc_vectors = embeddings_model.embed_documents(text)
# len(doc_vectors): 4개 문서 -> 4개 벡터
# len(doc_vectors[0]): 각 벡터는 1536차원
for vec in doc_vectors:
    print(len(vec))

In [ ]:
# 첫 번째 문서 벡터 값 일부 확인
doc_vectors[0][:5]

## 5. Cosine Similarity - 벡터 간 유사도

### 코사인 유사도란?

> **비유**: 나침반의 두 바늘이 같은 방향을 가리키면 1.0, 정반대면 -1.0, 직각(무관)이면 0. 벡터도 똑같이 "방향이 얼마나 일치하는가"로 유사도를 잰다.

- 두 벡터의 **사잇각(θ)의 코사인 값**
- `cos(0°) = 1` → 완전 같은 방향 (유사도 최대)
- `cos(90°) = 0` → 직교 (무관)
- `cos(180°) = -1` → 정반대 방향
- 벡터의 **크기(길이)는 무시**하고 **방향**만 봄 - 문장 길이에 덜 민감

수식: `cos(θ) = (A · B) / (|A| × |B|)`

In [ ]:
# sklearn에서 코사인 유사도 함수 임포트
# !pip install scikit-learn
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 4개 문서 간 모든 쌍의 유사도 계산 -> 4x4 대칭 행렬
# 대각선은 자기 자신과의 유사도라 모두 1.0
# [0][1]: "AI가 세상을 바꾸고 있다" vs "AI는 인공지능이다" 유사도
cosine_similarity(doc_vectors)

In [ ]:
# 쿼리 벡터를 맨 앞에 추가해서 '쿼리 vs 각 문서' 유사도를 한 번에
all_vectors = [query_vector] + doc_vectors
all_texts = [query] + text

cosine_similarity(all_vectors)
# 인공지능이 세상을 바꾸고 있습니다 -> AI가 세상을 바꾸고 있습니다 가장 유사함

In [ ]:
# pandas DataFrame으로 예쁘게 출력
similarity_matrix = cosine_similarity(all_vectors)
labels = ['query'] + [f'doc{i+1}' for i in range(len(text))]

df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
print(df)

### 주의: 코사인 유사도는 '확률'이 아니다!

옛날엔 "query와 doc1 유사도가 77.4%"처럼 해석하는 분들도 많았지만,
이건 **확률이 아니라 두 벡터의 방향 일치도**를 나타내는 스칼라일 뿐이다.

실무에서는 threshold(예: 0.5 이상만 유효)를 두고 필터링하거나,
그 이하면 외부 검색 툴로 넘기는 식으로 활용한다.

## 6. 실습: 텍스트 카테고리 분류기

각 카테고리(기술/스포츠/음식)의 예문들을 임베딩해서 **평균 벡터**를 만든 뒤,
새 문장의 임베딩과 비교해 가장 가까운 카테고리를 찾는다.

> **비유**: 교실마다 학생들의 평균 키를 구해 놓고, 새 학생이 왔을 때 "이 반 평균 키랑 제일 비슷하니까 네 반이야"라고 배정하는 느낌.

In [ ]:
# 질문이 어떤 카테고리와 가장 유사한지 분류하는 함수를 작성해 보세요.
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}

# [내 풀이] 각 카테고리 예문들을 임베딩만 저장하고, 쿼리와의 평균 유사도로 판정
categories_vectors = {}
for label, examples in categories.items():
    categories_vectors[label] = embeddings_model.embed_documents(examples)

def classify(text, categories_vectors, embeddings_model):
    # 쿼리 임베딩
    query_vector = embeddings_model.embed_query(text)

    best_category = None
    max_similarity = -1

    for label, example_vectors in categories_vectors.items():
        # 쿼리와 카테고리 내 모든 예문 벡터들 간 유사도 리스트
        similarities = cosine_similarity([query_vector], example_vectors)[0]
        # 카테고리의 평균 유사도로 비교
        avg_similarity = np.mean(similarities)

        if avg_similarity > max_similarity:
            max_similarity = avg_similarity
            best_category = label

    return best_category

query = '프로야구 시즌은?'
print(classify(query, categories_vectors, embeddings_model))

In [ ]:
# [선생님 답안] 예문 벡터의 '평균'을 카테고리 대표 벡터로 만들고 비교
# np.mean(embs, axis=0) -> 여러 벡터의 요소별 평균 = 카테고리 센트로이드
def classify(text, categories):
    cat_vectors = {}
    for cat, examples in categories.items():
        embs = embeddings_model.embed_documents(examples)
        cat_vectors[cat] = np.mean(embs, axis=0)  # 축 0 방향 평균 = 평균 벡터

    # 입력 텍스트 임베딩
    text_emb = embeddings_model.embed_query(text)

    # 각 카테고리 평균 벡터와 유사도 계산
    scores = {}
    for cat, cat_vec in cat_vectors.items():
        sim = cosine_similarity([text_emb], [cat_vec])[0][0]
        scores[cat] = sim

    # 가장 높은 점수의 카테고리 선택
    best_cat = max(scores, key=scores.get)

    print(f"입력 : {text}")
    print(f"예측 : {best_cat}")
    for c, s in scores.items():
        marker = " <<<" if c == best_cat else ""
        print(f" {c}: {s}{marker}")

    return best_cat, scores

test_sentense = ['새로운 GPU가 출시되어 인터넷이 빨라집니다.', '올해 올림픽에서 금메달 따면 좋겠어요.']

for sent in test_sentense:
    classify(sent, categories)
    print('---')

In [ ]:
# 추가 테스트: 다양한 카테고리로 분류가 잘 되는지 확인
test_senteces = [
    "새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다",
    "올해 올림픽에서 한국이 좋은 성적을 거뒀다",
    "이 식당 불고기가 정말 맛있다"
]

for sent in test_senteces:
    print()
    classify(sent, categories)

### LLM 프롬프트 분류 vs 임베딩 분류 - 언제 무엇을 쓸까?

| 방식 | 장점 | 단점 |
|---|---|---|
| **프롬프트 + few-shot** | 구현 쉬움, 정확도 높음 가능 | LLM 호출 비용, 출력 포맷 관리 필요 |
| **임베딩 + cosine** | 빠르고 저렴, 결정론적(deterministic) | 임계값 튜닝 필요, 복잡한 의도 파악 한계 |

**결정론적(deterministic)이란?**
같은 입력을 100번, 1000번 넣어도 **항상 동일한 벡터**가 나옴 (모델이 업데이트되지 않는 한).
서비스 안정성 측면에서 중요한 속성.

## 7. Embedding Cache - 비용 절감

### 왜 캐싱?

- 임베딩 API도 호출마다 **토큰당 비용** 발생
- 사내 매뉴얼, FAQ 같은 자료는 거의 안 바뀜 → **같은 문장을 반복 임베딩하면 낭비**
- **CacheBackedEmbeddings**: 한 번 계산한 벡터를 저장해뒀다가 재사용

> **비유**: 자주 보는 책은 도서관에서 빌리지 않고 집 책장에 꽂아두는 것. 두 번째 볼 땐 도서관 갈 필요 X.

**구조**:
- `InMemoryByteStore`: RAM에 캐시 저장 (프로세스 종료 시 날아감)
- 영구 저장이 필요하면 SQLite 등 ByteStore 구현체 교체
- 내부적으로 텍스트의 SHA-1 해시를 키로 사용

In [ ]:
# Embedding Cache "CacheBackedEmbeddings"
# langchain_classic.embeddings 에 위치 (최신 버전)
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

In [ ]:
# 1) InMemory ByteStore 생성 (RAM 저장소)
store = InMemoryByteStore()
# 2) CacheBackedEmbeddings: 기존 embedding_model을 감싸 캐시 기능 부여
#    namespace='embedding-cache' - 여러 모델을 한 store에 둘 때 충돌 방지
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embedding_model, store, namespace='embedding-cache'
)
# SHA-1 경고는 무시 OK (캐시 키용 해시라 보안 공격 위험은 극히 제한적)

## 정리

오늘 배운 흐름:

```
텍스트 --tokenizer(BPE)--> [토큰 ID] --embedding model--> [1536차원 벡터]
                                                              ↓
                                                         cosine similarity
                                                              ↓
                                               유사도/검색/카테고리 분류
```

**핵심 개념 요약**:
- **토큰**: LLM의 언어 단위. 한글 1글자 ≈ 2~3 토큰
- **임베딩**: 텍스트를 고차원 벡터로 변환 = 모델의 세계관 속 좌표
- **청킹**: 긴 문서를 context window에 맞게 쪼개기 (RecursiveCharacterTextSplitter 권장)
- **코사인 유사도**: 벡터 방향 일치도로 의미적 유사성 측정
- **임베딩 캐시**: 반복 임베딩 비용/시간 절약

**다음 시간 (3/18)**:
- 임베딩 캐시 실전 성능 비교
- 배치 처리 함수
- FAISS (Facebook AI Similarity Search) 벡터 스토어
- PCA/t-SNE로 임베딩 시각화
- 다국어 임베딩